# Cell 0: Install Dependencies


In [ ]:
# ── Cell 0: Install ONNX Runtime + TF 2.20 + scipy ──────────────────
# BirdCLEF+ 2026 Integrated Solution — combines 2026/2025/2024 top solutions
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

# ONNX Runtime — 150x faster than TF SavedModel
ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")

# TensorFlow 2.20 (needed for SavedModel fallback)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
print("TF 2.20 installed")

# scipy — for Gaussian smoothing (Technique 6 from 2025 1st place)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy"], check=True)
print("scipy installed")

try:
    import onnxruntime as ort
    _ONNX_AVAILABLE = True
    print("ONNX Runtime available ✅")
except ImportError:
    _ONNX_AVAILABLE = False
    print("ONNX not available, falling back to TF")


In [ ]:
# ── Cell 1: Mode switch ──────────────────────────────────────────────
# "submit" = inference only (for Kaggle submission)
# "train"  = local CV with OOF evaluation
MODE = "submit"

assert MODE in {"train", "submit"}
print("MODE =", MODE)


In [ ]:
# ── Cell 2: Imports & Comprehensive Configuration ────────────────────
# Integrates hyperparameters from 2026 current + 2024 1st + 2025 1st solutions
import os, re, gc, time, warnings, math
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

try:
    from scipy.ndimage import convolve1d
    _SCIPY_AVAILABLE = True
except ImportError:
    _SCIPY_AVAILABLE = False
    print("WARNING: scipy not available, Gaussian smoothing disabled")

tf.experimental.numpy.experimental_enable_numpy_behavior()
try: tf.config.set_visible_devices([], "GPU")
except: pass

_WALL_START = time.time()

BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
WORK_DIR  = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12          # 12 × 5s = 60s per file

CFG = {
    # ── Inference ────────────────────────────────────────────────────
    "batch_files": 16,

    # ── Local CV ─────────────────────────────────────────────────────
    "oof_n_splits": 5 if MODE == "train" else 3,

    # ── Dry-run ─────────────────────────────────────────────────────
    "dryrun_n_files": 20 if MODE == "train" else 0,

    # ── Train-only flags ────────────────────────────────────────────
    "run_oof": MODE == "train",
    "verbose": MODE == "train",

    # ══════════════════════════════════════════════════════════════════
    # ENHANCED: 2026 Current Solution Parameters
    # ══════════════════════════════════════════════════════════════════
    "proto_ssm_train": {
        "n_epochs":        80 if MODE == "train" else 40,
        "lr":              8e-4,
        "weight_decay":    1e-3,
        "val_ratio":       0.15,
        "patience":        25 if MODE == "train" else 12,
        "pos_weight_cap":  25.0,
        "distill_weight":  0.15,
        "proto_margin":    0.15,
        "label_smoothing": 0.03,
        "oof_n_splits":    5 if MODE == "train" else 3,
        "mixup_alpha":     0.4,
        "focal_gamma":     2.5,
        # SWA (Stochastic Weight Averaging) from current solution
        "swa_start_frac":  0.65,
        "swa_lr":          4e-4,
        "use_cosine_restart": True,
        "restart_period":  20,
        # NEW: Pseudo-labeling rounds (Technique 2 from 2024 1st)
        "pseudo_label_rounds": 2,
        "pseudo_label_threshold": 0.70,
        "pseudo_label_use_soundscapes": True,
    },
    "residual_ssm": {
        "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
        "dropout": 0.1, "correction_weight": 0.30,
        "n_epochs": 40 if MODE == "train" else 20,
        "lr": 8e-4,
        "patience": 12 if MODE == "train" else 6,
    },
    "mlp_params": {
        "hidden_layer_sizes": (256, 128), "activation": "relu",
        "max_iter": 600 if MODE == "train" else 300,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20 if MODE == "train" else 10,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    },

    # ══════════════════════════════════════════════════════════════════
    # NEW: 2024 1st Place Technique Parameters
    # ══════════════════════════════════════════════════════════════════
    # Technique 1: Context window concatenation for SED mel
    "context_window": True,

    # Technique 3: Temporal moving average smoothing
    "moving_avg_smooth": True,
    "moving_avg_kernel": 5,

    # ══════════════════════════════════════════════════════════════════
    # NEW: 2025 1st Place Technique Parameters
    # ══════════════════════════════════════════════════════════════════
    # Technique 4: Full-signal mel spectrogram
    "full_signal_mel": True,

    # Technique 5: Overlap-average-max-delta TTA for SED
    "overlap_delta_tta": True,

    # Technique 6: Gaussian temporal smoothing
    "gaussian_smooth": True,

    # Technique 7: Cross-taxonomy species enhancement
    "cross_taxonomy_enhance": True,

    # Technique 8: Power scaling for ensemble
    "ensemble_power": 1.0,

    # ── SED mel parameters (shared) ─────────────────────────────────
    "n_mels_sed": 256,
    "n_fft_sed": 2048,
    "hop_sed": 512,
    "fmin_sed": 20,
    "fmax_sed": 16000,
    "top_db_sed": 80,

    # ── Ensemble weights ────────────────────────────────────────────
    "proto_weight": 0.65,
    "sed_weight": 0.35,
    "mlp_alpha_blend": 0.4,
    "first_pass_ensemble_w": 0.5,

    # ── Post-processing chain ───────────────────────────────────────
    "prior_lambda": 0.4,
    "file_conf_top_k": 2,
    "file_conf_power": 0.4,
    "rank_aware_power": 0.4,
    "adaptive_delta_base_alpha": 0.20,

    # ── Temperature scaling ─────────────────────────────────────────
    "bird_temp": 1.10,
    "texture_temp": 0.95,
}

print("✅ Integrated CFG loaded")
print(f"  n_epochs={CFG['proto_ssm_train']['n_epochs']}  "
      f"patience={CFG['proto_ssm_train']['patience']}  "
      f"oof_n_splits={CFG['proto_ssm_train']['oof_n_splits']}  "
      f"mlp_max_iter={CFG['mlp_params']['max_iter']}")
print(f"  NEW 2024: context_window={CFG['context_window']}  "
      f"moving_avg_kernel={CFG['moving_avg_kernel']}")
print(f"  NEW 2025: full_signal_mel={CFG['full_signal_mel']}  "
      f"overlap_delta_tta={CFG['overlap_delta_tta']}  "
      f"gaussian_smooth={CFG['gaussian_smooth']}  "
      f"ensemble_power={CFG['ensemble_power']}")
print(f"  Ensemble: ProtoSSM {CFG['proto_weight']:.0%} + SED {CFG['sed_weight']:.0%}")
print(f"  run_oof={CFG['run_oof']}  verbose={CFG['verbose']}  dryrun={CFG['dryrun_n_files']}")


In [ ]:
# ── Cell 3: Data Loading & Label Parsing ────────────────────────────
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

sc = (soundscape_labels
      .groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels)
      .reset_index(name="label_list"))

sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)

_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)

full_rows = (sc[sc["fully_labeled"]]
             .sort_values(["filename", "end_sec"])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

# Class taxonomy map for per-taxon temperature and cross-taxonomy enhancement
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}  # continuous sound callers
AVES_TAXA      = {"Aves"}

# Identify texture-caller species (for Technique 7: cross-taxonomy enhancement)
TEXTURE_SPECIES = [i for i, lbl in enumerate(PRIMARY_LABELS)
                  if CLASS_NAME_MAP.get(lbl) in TEXTURE_TAXA]
AVES_SPECIES = [i for i, lbl in enumerate(PRIMARY_LABELS)
               if CLASS_NAME_MAP.get(lbl) in AVES_TAXA]

print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")
print(f"Texture callers (amphibians+insects): {len(TEXTURE_SPECIES)}")
print(f"Aves (birds): {len(AVES_SPECIES)}")


In [ ]:
# ── Cell 4: Load PERCH Model (ONNX) + Species Mapping ──────────────
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn       = birdclassifier.signatures["serving_default"]

# ONNX session (150x faster than TF SavedModel)
ONNX_PERCH_PATH = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx")
USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()

if USE_ONNX:
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                            providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print("Using ONNX Perch (150x faster)")
else:
    print("Using TF SavedModel Perch")

bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL = len(bc_labels)

mapping = (taxonomy
           .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                  on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")

# ── Genus proxy logits for unmapped species ─────────────────────────
# For species without direct PERCH mapping, use genus-level matches
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)

proxy_map = {}   # label_idx -> list of bc_indices
unmapped_df = (taxonomy[taxonomy["primary_label"]
               .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy())

for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci    = str(row["scientific_name"])
    genus  = sci.split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ]
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

# Only use proxies for biologically meaningful taxa
PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map  = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}

print(f"Unmapped species total:        {len(UNMAPPED_POS)}")
print(f"Species with genus proxy:      {len(proxy_map)}")
print(f"Species still without signal:  {len(UNMAPPED_POS) - len(proxy_map)}")


In [ ]:
# ── Cell 5: Perch Inference Engine (ONNX + Multithreaded I/O) ───────
# Extracts both logits (N_CLASSES) and 1536-dim embeddings
import concurrent.futures

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:                      y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=16, verbose=True):
    """Run PERCH inference on a list of audio files.
    Returns: (meta_df, scores, embeddings)
        meta_df: DataFrame with row_id, filename, site, hour_utc
        scores:  (n_rows, N_CLASSES) float32 — raw PERCH logits
        embs:    (n_rows, 1536) float32 — embedding vectors
    """
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS

    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536),      dtype=np.float32)

    wr  = 0
    itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        next_paths   = paths[0:batch_files]
        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

        for start in itr:
            batch_paths  = next_paths
            batch_n      = len(batch_paths)
            batch_audio  = [f.result() for f in future_audio]

            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr

            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta["site"]
                hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS

            # ONNX or TF inference
            if USE_ONNX:
                outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            else:
                out    = infer_fn(inputs=tf.convert_to_tensor(x))
                logits = out["label"].numpy().astype(np.float32)
                emb    = out["embedding"].numpy().astype(np.float32)

            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb

            # Genus proxy for unmapped species
            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)

            del x, logits, emb, batch_audio
            gc.collect()

    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                             "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

print("✅ Perch inference engine defined")


In [ ]:
# ── Cell 6: Build/Load Perch Training Cache ────────────────────────
print(f"USE_ONNX = {USE_ONNX}")

EXTERNAL_CACHE_DIRS = [
    Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
    Path("/kaggle/input/datasets/jaejohn/perch-meta"),
]

CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"


def _find_external_cache():
    for d in EXTERNAL_CACHE_DIRS:
        meta = d / "perch_meta.parquet"
        npz  = d / "perch_arrays.npz"
        if meta.exists() and npz.exists():
            return meta, npz
    return None, None


SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]


def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k
    raise KeyError(f"None of {candidates} found in npz. Available: {arr.files}")


def _build_cache():
    print(f"Building Perch cache from {len(full_files)} training files…")
    train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    train_paths = [p for p in train_paths if p.exists()]
    t0 = time.time()
    meta_built, sc_built, emb_built = run_perch(
        train_paths, batch_files=CFG["batch_files"], verbose=True)
    print(f"  Perch pass done in {time.time()-t0:.1f}s  scores={sc_built.shape} embs={emb_built.shape}")
    meta_built.to_parquet(CACHE_META_LOCAL)
    np.savez(CACHE_NPZ_LOCAL,
             scores=sc_built.astype(np.float32),
             embs=emb_built.astype(np.float32),
             primary_labels=np.array(PRIMARY_LABELS))
    print(f"  Cache saved to {WORK_DIR}")
    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL


ext_meta, ext_npz = _find_external_cache()
if ext_meta is not None:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
    print(f"Using external cache: {CACHE_META.parent}")
elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
    print(f"Using local cache: {WORK_DIR}")
else:
    print("No cache found — building from scratch (~1.5 min)")
    CACHE_META, CACHE_NPZ = _build_cache()

print("Loading Perch cache…")
meta_tr = pd.read_parquet(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)
print(f"  scores ← '{sk}'  shape={sc_tr_raw.shape}")
print(f"  embs   ← '{ek}'  shape={emb_tr_raw.shape}")

sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)

# Reconstruct row_id if missing
if "row_id" not in meta_tr.columns:
    if "end_sec" in meta_tr.columns:
        end_sec = meta_tr["end_sec"].astype(int)
    elif "window_idx" in meta_tr.columns:
        end_sec = (meta_tr["window_idx"].astype(int) + 1) * 5
    else:
        end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)
    meta_tr["row_id"] = (
        meta_tr["filename"].str.replace(".ogg", "", regex=False)
        + "_" + end_sec.astype(str))

row_id_to_index = full_rows.set_index("row_id")["index"]
missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)
if missing_rows:
    raise RuntimeError(
        f"Cache has {len(missing_rows)} row_ids not in labeled set. "
        f"Delete cache files and rebuild.")

Y_FULL_aligned = Y_SC[
    row_id_to_index.loc[meta_tr["row_id"]].to_numpy()
]
print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}")


In [ ]:
# ── Cell 7: Metric Helpers ──────────────────────────────────────────
def macro_auc(y_true, y_score):
    """Exact replica of competition metric: macro-averaged ROC-AUC,
    skipping classes with no positive labels."""
    keep = y_true.sum(axis=0) > 0
    if keep.sum() == 0:
        return 0.0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")


def honest_oof_auc(scores, Y, meta_df, n_splits=5, label="scores"):
    """GroupKFold by filename — files never split across folds.
    This is the only correct way to estimate LB performance locally."""
    groups = meta_df["filename"].to_numpy()
    gkf    = GroupKFold(n_splits=n_splits)
    oof    = np.zeros_like(scores, dtype=np.float32)
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(scores, groups=groups), 1):
        oof[va_idx] = scores[va_idx]
    auc = macro_auc(Y, oof)
    print(f"[{label}] honest OOF macro-AUC: {auc:.6f}")
    return auc, oof

print("✅ Metric helpers defined")


In [ ]:
# ── Cell 8: Post-Processing Functions (ENHANCED) ────────────────────
# Combines all post-processing from 2026 current + 2024/2025 1st place

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


# ════════════════════════════════════════════════════════════════════
# FROM 2026 CURRENT SOLUTION
# ════════════════════════════════════════════════════════════════════

def smooth_predictions(probs, n_windows=12, alpha=0.3):
    """Fixed-alpha temporal smoothing. new[t] = (1-a)*old[t] + 0.5*a*(old[t-1]+old[t+1])"""
    N, C = probs.shape
    assert N % n_windows == 0
    view = probs.reshape(-1, n_windows, C).copy()
    prev_w = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)
    next_w = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)
    smoothed = (1 - alpha) * view + 0.5 * alpha * (prev_w + next_w)
    return smoothed.reshape(N, C)


def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    """[2026] Confidence-adaptive temporal smoothing.
    Alpha inversely proportional to max prediction confidence.
    Confident windows preserved, noisy windows smoothed."""
    N, C = probs.shape
    assert N % n_windows == 0
    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)
    out    = result.reshape(-1, n_windows, C)

    for t in range(n_windows):
        conf  = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)

        if t == 0:
            neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0

        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg

    return result


def build_prior_tables(sc_df, Y_labels):
    """[2026] Build site-level and hour-level species frequency tables
    with Bayesian shrinkage toward global average."""
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)

    site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]
        mask = sc_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_labels[mask].mean(axis=0)

    hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]
        mask = sc_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_labels[mask].mean(axis=0)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i, "site_p": site_p, "site_n": site_n,
        "hour_to_i": hour_to_i, "hour_p": hour_p, "hour_n": hour_n,
    }


def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    """[2026] Add Bayesian-shrunk prior logit to raw Perch scores."""
    eps = 1e-4
    n   = len(scores)
    out = scores.copy()
    p = np.tile(tables["global_p"], (n, 1))

    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j = tables["hour_to_i"][h]
            nh = tables["hour_n"][j]
            w  = nh / (nh + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]

    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j = tables["site_to_i"][s]
            ns = tables["site_n"][j]
            w  = ns / (ns + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]

    p = np.clip(p, eps, 1 - eps)
    logit_prior = np.log(p) - np.log1p(-p)
    out += lambda_prior * logit_prior
    return out.astype(np.float32)


def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.4):
    """[2026] Scale predictions by file-level confidence (top-k mean × power)."""
    N, C = probs.shape
    assert N % n_windows == 0
    view       = probs.reshape(-1, n_windows, C)
    sorted_v   = np.sort(view, axis=1)
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
    scale      = np.power(top_k_mean, power)
    return (view * scale).reshape(N, C)


def rank_aware_scaling(probs, n_windows=12, power=0.4):
    """[2026] Scale by file-max confidence × power.
    Asks: does ANY window have strong evidence?"""
    N, C = probs.shape
    assert N % n_windows == 0
    view     = probs.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)
    scale    = np.power(file_max, power)
    return (view * scale).reshape(N, C)


# ════════════════════════════════════════════════════════════════════
# NEW FROM 2024 1st PLACE
# ════════════════════════════════════════════════════════════════════

def moving_average_smooth(probs, n_windows=12, kernel_size=5):
    """[2024 1st] Temporal moving-average smoothing.
    Applies 1D convolution along time axis with uniform kernel.
    Applied BEFORE adaptive delta smoothing in the chain."""
    N, C = probs.shape
    assert N % n_windows == 0
    view = probs.reshape(-1, n_windows, C).copy()
    kernel = np.ones(kernel_size) / kernel_size

    for f in range(view.shape[0]):
        for c in range(C):
            view[f, :, c] = np.convolve(view[f, :, c], kernel, mode='same')

    return view.reshape(N, C)


# ════════════════════════════════════════════════════════════════════
# NEW FROM 2025 1st PLACE
# ════════════════════════════════════════════════════════════════════

def gaussian_smooth(probs, axis=0):
    """[2025 1st] Gaussian kernel smoothing per-file.
    Kernel: [0.1, 0.2, 0.4, 0.2, 0.1] applied along time axis WITHIN each file
    (no cross-file leakage)."""
    if not _SCIPY_AVAILABLE:
        return probs
    N, C = probs.shape
    n_files = N // N_WINDOWS
    view = probs.reshape(n_files, N_WINDOWS, C).copy()
    gaussian_kernel = np.array([0.1, 0.2, 0.4, 0.2, 0.1])
    for c in range(C):
        view[:, :, c] = convolve1d(
            view[:, :, c], gaussian_kernel, axis=1, mode='nearest')
    return view.reshape(N, C).astype(np.float32)


def per_taxon_temperature(scores):
    """[2026] Per-taxon temperature scaling.
    Birds T=1.10 (softer), frogs/insects T=0.95 (sharper)."""
    temperatures = np.ones(N_CLASSES, dtype=np.float32)
    for ci, label in enumerate(PRIMARY_LABELS):
        cls = CLASS_NAME_MAP.get(label, "Aves")
        if cls in TEXTURE_TAXA:
            temperatures[ci] = CFG["texture_temp"]
        else:
            temperatures[ci] = CFG["bird_temp"]
    return scores / temperatures[None, :]


def power_scale_blend(preds1, preds2, w1, w2, power=1.0):
    """[2025 1st] Power-scaled ensemble blend.
    blended = w1 * (preds1 ** power) + w2 * (preds2 ** power)"""
    eps = 1e-7
    p1 = np.clip(preds1, eps, 1.0)
    p2 = np.clip(preds2, eps, 1.0)
    blended = w1 * (p1 ** power) + w2 * (p2 ** power)
    return blended.astype(np.float32)


print("✅ All post-processing functions defined")
print("  [2026] adaptive_delta, file_confidence, rank_aware, prior_tables")
print("  [2024] moving_average_smooth")
print("  [2025] gaussian_smooth, power_scale_blend")


In [ ]:
# ── Cell 9: Full-Signal Mel Spectrogram Processor ───────────────────
# [2025 1st Place] Technique 4: Compute mel from full 60s audio,
# then slice into overlapping windows. Avoids boundary artifacts.
# [2024 1st Place] Technique 1: Context window concatenation —
# each chunk augmented with ±5s context from neighboring chunks.

import librosa

N_MELS_SED = CFG["n_mels_sed"]
N_FFT_SED  = CFG["n_fft_sed"]
HOP_SED    = CFG["hop_sed"]
FMIN_SED   = CFG["fmin_sed"]
FMAX_SED   = CFG["fmax_sed"]
TOP_DB_SED = CFG["top_db_sed"]


def compute_full_signal_mel(wave, sr=32000):
    """[2025 1st] Compute mel spectrogram from full 60s audio.
    Returns: (n_mels, time_frames) float32 array."""
    mel = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=N_FFT_SED, hop_length=HOP_SED,
        n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED,
        power=2.0
    )
    mel_db = librosa.power_to_db(mel, top_db=TOP_DB_SED)
    return mel_db.astype(np.float32)


def slice_full_mel_into_windows(mel_full, n_windows=12, sr=32000,
                                  window_sec=5, stride_sec=5):
    """[2025 1st] Slice full-signal mel into overlapping windows.
    Each window corresponds to 5s of audio at the given stride.
    Returns: (n_windows, n_mels, time_frames_per_window) float32."""
    samples_per_window = sr * window_sec
    hop_samples = sr * stride_sec
    # librosa melspectrogram uses center=True by default,
    # which produces (samples // hop) + 1 frames
    frames_per_window = samples_per_window // HOP_SED + 1
    hop_frames = hop_samples // HOP_SED

    total_frames = mel_full.shape[1]
    windows = []

    for i in range(n_windows):
        start_frame = i * hop_frames
        end_frame = start_frame + frames_per_window
        if end_frame > total_frames:
            # Pad if needed
            chunk = np.zeros((mel_full.shape[0], frames_per_window), dtype=np.float32)
            available = total_frames - start_frame
            chunk[:, :available] = mel_full[:, start_frame:total_frames]
        else:
            chunk = mel_full[:, start_frame:end_frame]
        windows.append(chunk)

    return np.stack(windows)  # (n_windows, n_mels, frames)


def apply_context_window_concat(mel_windows, trim_ratio=0.5):
    """[2024 1st] Context window concatenation for mel-based models.
    Each chunk sees ±5s of context from neighboring chunks, then trimmed.
    chunks_1 = np.concatenate([chunks[:1], chunks[:-1]], axis=0)  # previous
    chunks_2 = np.concatenate([chunks[1:], chunks[-1:]], axis=0)  # next
    chunks = np.concatenate([chunks_1, chunks, chunks_2], axis=-1)  # concat time
    Then trim center portion.
    Input:  (n_windows, n_mels, frames)
    Output: (n_windows, n_mels, frames) — same shape, but with context"""
    if not CFG["context_window"]:
        return mel_windows

    n_win, n_mels, frames = mel_windows.shape

    # Previous window (repeat first for boundary)
    chunks_prev = np.concatenate([mel_windows[:1], mel_windows[:-1]], axis=0)
    # Next window (repeat last for boundary)
    chunks_next = np.concatenate([mel_windows[1:], mel_windows[-1:]], axis=0)

    # Concatenate along time axis: [prev, current, next]
    concat = np.concatenate([chunks_prev, mel_windows, chunks_next], axis=-1)
    # concat shape: (n_windows, n_mels, 3 * frames)

    # Trim center portion to get back to original frame count
    total_frames = concat.shape[-1]
    trim_start = frames  # skip the 'prev' portion
    trimmed = concat[:, :, trim_start:trim_start + frames]

    return trimmed


def normalize_mel(mel_windows):
    """Per-chunk z-score normalization."""
    result = np.zeros_like(mel_windows)
    for i in range(mel_windows.shape[0]):
        chunk = mel_windows[i]
        mean = chunk.mean()
        std = chunk.std() + 1e-6
        result[i] = (chunk - mean) / std
    return result


def file_to_sed_mel(path, sr=32000):
    """Full pipeline: read audio → full-signal mel → slice → context → normalize.
    Returns: (n_windows, 1, n_mels, frames) ready for SED ONNX model."""
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr0 != sr:
        y = librosa.resample(y, orig_sr=sr0, target_sr=sr)

    n_samples = 60 * sr
    if len(y) < n_samples:
        y = np.pad(y, (0, n_samples - len(y)))
    else:
        y = y[:n_samples]

    if CFG["full_signal_mel"]:
        # [2025 1st] Compute mel from full 60s signal
        mel_full = compute_full_signal_mel(y, sr=sr)
        mel_windows = slice_full_mel_into_windows(mel_full, N_WINDOWS, sr)
    else:
        # Fallback: per-chunk mel
        chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
        mel_windows = np.stack([
            librosa.feature.melspectrogram(
                y=c, sr=sr, n_fft=N_FFT_SED, hop_length=HOP_SED,
                n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0
            ) for c in chunks
        ])
        mel_windows = librosa.power_to_db(mel_windows, top_db=TOP_DB_SED).astype(np.float32)

    # [2024 1st] Context window concatenation
    mel_windows = apply_context_window_concat(mel_windows)

    # Normalize
    mel_windows = normalize_mel(mel_windows)

    # Add channel dimension: (n_windows, 1, n_mels, frames)
    return mel_windows[:, None, :, :].astype(np.float32)


print("✅ Full-signal mel processor defined")
print(f"  [2025 1st] full_signal_mel={CFG['full_signal_mel']}")
print(f"  [2024 1st] context_window={CFG['context_window']}")


In [ ]:
# ── Cell 10: SED Inference with Overlap-Delta TTA ───────────────────
# [2026] 5-fold Tucker Arrants SED ONNX models
# [2025 1st] Overlap-average-max-delta temporal TTA
# [2025 1st] Gaussian smoothing on SED output
# [2025 1st] Technique 7: Cross-taxonomy species enhancement


def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError(
            "sed_fold0.onnx not found. "
            "Attach tuckerarrants/bc2026-distilled-sed-public.")
    return hits[0].parent


def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so,
                                providers=["CPUExecutionProvider"])


def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)


def overlap_average_max_delta_tta(frame_preds, shift_frames=2):
    """[2025 1st] Overlap-Average-Max-Delta Temporal TTA.
    - 50% weight from original segment max
    - 25% weight from segment shifted 2 frames backward
    - 25% weight from segment shifted 2 frames forward"""
    if not CFG["overlap_delta_tta"]:
        return frame_preds

    n_windows, n_classes = frame_preds.shape

    # Original (max-aggregated)
    original = frame_preds.copy()

    # Shift backward by 2 frames
    shifted_bwd = np.roll(frame_preds, shift_frames, axis=0)
    shifted_bwd[:shift_frames] = frame_preds[:shift_frames]  # boundary handling

    # Shift forward by 2 frames
    shifted_fwd = np.roll(frame_preds, -shift_frames, axis=0)
    shifted_fwd[-shift_frames:] = frame_preds[-shift_frames:]  # boundary handling

    # Weighted combination
    combined = 0.50 * original + 0.25 * shifted_bwd + 0.25 * shifted_fwd

    return combined.astype(np.float32)


# Load the 5 SED fold models
sed_dir = find_sed_dir()
sed_fold_paths = sorted(
    sed_dir.glob("sed_fold*.onnx"),
    key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1))
)
sed_sessions = [make_sed_session(p) for p in sed_fold_paths]

print(f"SED dir: {sed_dir}")
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")


def run_sed_inference(test_paths, verbose=True):
    """Run SED inference on all test files with enhanced TTA.
    Returns: (row_ids, predictions_array)"""
    sed_rows = []
    sed_preds = []
    _t0 = time.time()

    for i, path in enumerate(test_paths, 1):
        mel = file_to_sed_mel(path)  # (12, 1, n_mels, frames)

        # Average predictions across all folds
        p_sum = np.zeros((N_WINDOWS, N_CLASSES), dtype=np.float32)
        for sess in sed_sessions:
            outs = sess.run(None, {sess.get_inputs()[0].name: mel})
            clip_logits = outs[0]              # (12, 234)
            frame_max   = outs[1].max(axis=1)  # (12, 234)
            p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)

        p_mean = p_sum / len(sed_sessions)

        # [2025 1st] Overlap-average-max-delta temporal TTA
        if CFG["overlap_delta_tta"]:
            p_mean = overlap_average_max_delta_tta(p_mean, shift_frames=2)

        # [2025 1st] Gaussian smoothing on SED output
        if CFG["gaussian_smooth"] and _SCIPY_AVAILABLE:
            from scipy.ndimage import gaussian_filter1d
            p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode='nearest').astype(np.float32)

        # [2025 1st] Technique 7: Cross-taxonomy species enhancement
        # Boost texture-caller (amphibian/insect) predictions slightly
        if CFG["cross_taxonomy_enhance"] and len(TEXTURE_SPECIES) > 0:
            # Mild boost for texture callers (they tend to be underpredicted)
            texture_boost = 1.05  # 5% boost
            p_mean[:, TEXTURE_SPECIES] = np.clip(
                p_mean[:, TEXTURE_SPECIES] * texture_boost, 0.0, 1.0)

        stem = path.stem
        ends = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
        sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
        sed_preds.append(p_mean)

        if i == 1 or i % 50 == 0 or i == len(test_paths):
            print(f"  SED: {i}/{len(test_paths)} | {time.time()-_t0:.1f}s")

    sed_preds_arr = np.concatenate(sed_preds, axis=0)
    return sed_rows, np.clip(sed_preds_arr, 0.0, 1.0)


print("✅ SED inference pipeline defined")
print(f"  [2025] overlap_delta_tta={CFG['overlap_delta_tta']}")
print(f"  [2025] gaussian_smooth={CFG['gaussian_smooth']}")
print(f"  [2025] cross_taxonomy_enhance={CFG['cross_taxonomy_enhance']}")


In [ ]:
# ── Cell 11: MLP Probes with PCA + Sequential Features ─────────────
# [2026] Per-class classifiers with PCA(64) embeddings + sequential features
# Vectorized inference via torch.bmm for speed
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier


def build_class_freq_weights(Y, cap=10.0):
    total     = Y.shape[0]
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / total
    weights   = 1.0 / (freq ** 0.5)
    weights   = np.clip(weights, 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)


def build_sequential_features(scores_col, n_windows=12):
    """Build prev/next/mean/max/std sequential features for one class."""
    N = len(scores_col)
    assert N % n_windows == 0
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev.reshape(-1), next_.reshape(-1), mean, max_, std


def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    """Train per-class MLP probes on PCA embeddings + sequential features."""
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f"Embedding: {emb.shape} → PCA: {Z.shape}  "
          f"(variance: {pca.explained_variance_ratio_.sum():.2%})")

    class_weights = build_class_freq_weights(Y, cap=10.0)
    probe_models = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    print(f"Training MLP probes for {len(active)} species (>= {min_pos} pos windows)…")

    MAX_ROWS = 3000

    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y):
            continue

        prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
        X = np.hstack([
            Z, scores_raw[:, ci:ci+1],
            prev[:, None], next_[:, None],
            mean[:, None], max_[:, None], std[:, None],
        ])

        n_pos = int(y.sum()); n_neg = len(y) - n_pos
        pos_idx = np.where(y == 1)[0]
        w      = float(class_weights[ci])
        repeat = max(1, int(round(w * n_neg / max(n_pos, 1))))
        repeat = min(repeat, 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))

        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])

        clf = MLPClassifier(
            hidden_layer_sizes=CFG["mlp_params"]["hidden_layer_sizes"],
            activation=CFG["mlp_params"]["activation"],
            max_iter=CFG["mlp_params"]["max_iter"],
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=CFG["mlp_params"]["n_iter_no_change"],
            random_state=42,
            learning_rate_init=CFG["mlp_params"]["learning_rate_init"],
            alpha=CFG["mlp_params"]["alpha"],
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf

    print(f"Trained {len(probe_models)} MLP probes")
    return probe_models, scaler, pca, alpha_blend


# Vectorized MLP inference via torch.bmm
class VectorizedMLPProbes(nn.Module):
    """Batched PyTorch model for fast MLP probe inference."""
    def __init__(self, probe_models):
        super().__init__()
        self.valid_classes = sorted(probe_models.keys())
        V = len(self.valid_classes)
        if V == 0:
            self.weights = nn.ParameterList()
            self.biases  = nn.ParameterList()
            self.n_layers = 0
            return
        sample = probe_models[self.valid_classes[0]]
        self.n_layers = len(sample.coefs_)
        self.weights  = nn.ParameterList()
        self.biases   = nn.ParameterList()
        for layer_idx in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[layer_idx]
                          for c in self.valid_classes], axis=0)
            b = np.stack([probe_models[c].intercepts_[layer_idx]
                          for c in self.valid_classes], axis=0)
            self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append(nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False))

    def forward(self, x):
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1:
                h = torch.relu(h)
        return h.squeeze(-1)


def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models,
                                 scaler, pca, alpha_blend=0.4):
    """Vectorized MLP probe inference using torch.bmm — 10-50x faster."""
    if len(probe_models) == 0:
        return scores_test.copy()

    emb_s  = scaler.transform(emb_test)
    Z_test = pca.transform(emb_s).astype(np.float32)

    valid_classes = sorted(probe_models.keys())
    V = len(valid_classes)
    N = len(scores_test)

    raw  = scores_test[:, valid_classes].T          # (V, N)
    n_files = N // N_WINDOWS
    raw_view = raw.reshape(V, n_files, N_WINDOWS)
    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt  = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1)
    mx   = np.repeat(raw_view.max(axis=2),  N_WINDOWS, axis=1)
    std  = np.repeat(raw_view.std(axis=2),  N_WINDOWS, axis=1)

    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)
    Z_expanded   = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))
    X_all = np.concatenate([Z_expanded.astype(np.float32), scalar_feats], axis=-1)

    vec_probe = VectorizedMLPProbes(probe_models)
    vec_probe.eval()
    with torch.no_grad():
        preds = vec_probe(torch.tensor(X_all)).numpy()

    result = scores_test.copy()
    base_valid = scores_test[:, valid_classes]
    result[:, valid_classes] = (1.0 - alpha_blend) * base_valid + alpha_blend * preds.T
    return result

print("✅ MLP probes defined (PCA(64) + sequential + vectorized inference)")


In [ ]:
# ── Cell 12: LightProtoSSM Model Definition ─────────────────────────
# [2026] Custom Selective SSM with prototype distillation,
# bidirectional scan, positional encoding, metadata injection (site + hour)

import torch
import torch.nn as nn
import torch.nn.functional as F


class SelectiveSSM(nn.Module):
    """Simplified Selective State Space Model core."""
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)
        h = torch.zeros(B_sz, D, self.d_state)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    """LightProtoSSM: Selective SSM + Prototype Distillation + Cross-Attention."""
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_classes=234, n_windows=12, dropout=0.15,
                 n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc  = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        # Bidirectional SSM layers
        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(2)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop      = nn.Dropout(dropout)

        if use_cross_attn:
            self.cross_attn  = nn.ModuleList([
                nn.MultiheadAttention(d_model, num_heads=cross_attn_heads,
                                      dropout=dropout, batch_first=True)
                for _ in range(2)])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])

        # Prototype layer with distillation
        self.prototypes   = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp   = nn.Parameter(torch.tensor(5.0))
        self.class_bias   = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]

        for i, (fwd, bwd, merge, norm) in enumerate(zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm)):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            if self.use_cross_attn:
                attn_out, _ = self.cross_attn[i](h, h, h)
                h = self.cross_norm[i](h + attn_out)

        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = (torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp)
               + self.class_bias[None, None, :])
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            out   = alpha * sim + (1 - alpha) * perch_logits
        else:
            out = sim
        return out

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("✅ LightProtoSSM model defined (SelectiveSSM + ProtoDistill + CrossAttn)")


In [ ]:
# ── Cell 13: ProtoSSM Training (ENHANCED with Pseudo-Labeling) ──────
# [2026] Initial training on labeled data with SWA
# [2024 1st] Technique 2: Semi-supervised pseudo-labeling —
#   After initial training, generate predictions on ALL train_soundscapes
#   (including unlabeled). Use high-confidence predictions (p > 0.7) as
#   additional pseudo-labels for a second training round.


def _prepare_file_level_data(emb_full, scores_full, Y_full, meta_full):
    """Helper: reshape flat arrays to file-level tensors.
    Uses positional grouping (every N_WINDOWS rows = 1 file)
    instead of filename.unique() to avoid mismatches."""
    n_files = len(emb_full) // N_WINDOWS
    trim = n_files * N_WINDOWS
    emb_f = emb_full[:trim].reshape(n_files, N_WINDOWS, -1)
    log_f = scores_full[:trim].reshape(n_files, N_WINDOWS, -1)
    lab_f = Y_full[:trim].reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    meta_trimmed = meta_full.iloc[:trim]
    sites_u = sorted(meta_full["site"].dropna().astype(str).unique())
    site2i  = {s: i + 1 for i, s in enumerate(sites_u)}

    # One site/hour per file from the first window row
    site_ids = np.zeros(n_files, dtype=np.int64)
    hour_ids = np.zeros(n_files, dtype=np.int64)
    for i in range(n_files):
        row = meta_trimmed.iloc[i * N_WINDOWS]
        site_ids[i] = min(site2i.get(str(row.get("site", "")), 0), 19)
        hour_ids[i] = int(row.get("hour_utc", 0)) % 24

    return emb_f, log_f, lab_f, site_ids, hour_ids, site2i


def _train_proto_ssm_epoch(model, emb_t, log_t, lab_t, site_t, hour_t,
                           pos_weight, opt, sched):
    """Single training epoch for ProtoSSM."""
    model.train()
    out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
    loss = (F.binary_cross_entropy_with_logits(
                out, lab_t, pos_weight=pos_weight[None, None, :])
            + CFG["proto_ssm_train"]["distill_weight"] * F.mse_loss(out, log_t))
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    return loss.item()


def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full,
                           n_epochs=40, patience=8, lr=1e-3,
                           n_sites=20, verbose=False):
    """Train LightProtoSSM with SWA + optional pseudo-labeling."""
    emb_f, log_f, lab_f, site_ids, hour_ids, site2i = \
        _prepare_file_level_data(emb_full, scores_full, Y_full, meta_full)

    model = LightProtoSSM(n_classes=N_CLASSES, n_sites=n_sites,
                          use_cross_attn=True, cross_attn_heads=2)
    model.init_prototypes(
        torch.tensor(emb_full, dtype=torch.float32),
        torch.tensor(Y_full,   dtype=torch.float32))
    print(f"LightProtoSSM params: {model.count_parameters():,}")

    emb_t  = torch.tensor(emb_f,    dtype=torch.float32)
    log_t  = torch.tensor(log_f,    dtype=torch.float32)
    lab_t  = torch.tensor(lab_f,    dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos_cnt    = lab_t.sum(dim=(0, 1))
    total      = lab_t.shape[0] * lab_t.shape[1]
    pos_weight = ((total - pos_cnt) / (pos_cnt + 1)).clamp(max=25.0)

    best_loss, best_state, wait = float("inf"), None, 0

    # SWA setup
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = int(n_epochs * CFG["proto_ssm_train"]["swa_start_frac"])
    swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=CFG["proto_ssm_train"]["swa_lr"])

    for ep in range(n_epochs):
        loss_val = _train_proto_ssm_epoch(
            model, emb_t, log_t, lab_t, site_t, hour_t, pos_weight, opt, sched)

        if ep >= swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            sched.step()

        if loss_val < best_loss:
            best_loss  = loss_val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    # Use SWA model if we collected enough checkpoints
    if ep >= swa_start:
        torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0), swa_model)
        model = swa_model
    else:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        out = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
    print(f"LightProtoSSM trained — best loss={best_loss:.4f}")
    return model, site2i


def generate_pseudo_labels(model, emb_full, scores_full, meta_full,
                           threshold=0.70, verbose=False):
    """[2024 1st] Generate pseudo-labels from model predictions.
    Only keep predictions with confidence > threshold."""
    model.eval()
    n_files = len(emb_full) // N_WINDOWS
    trim = n_files * N_WINDOWS
    emb_f = torch.tensor(
        emb_full[:trim].reshape(n_files, N_WINDOWS, -1), dtype=torch.float32)
    log_f = torch.tensor(
        scores_full[:trim].reshape(n_files, N_WINDOWS, -1), dtype=torch.float32)

    meta_trimmed = meta_full.iloc[:trim]
    site2i_local = {s: i + 1
                    for i, s in enumerate(sorted(
                        meta_full["site"].dropna().astype(str).unique()))}
    site_ids = torch.tensor([
        min(site2i_local.get(str(meta_trimmed.iloc[i * N_WINDOWS].get("site", "")), 0), 19)
        for i in range(n_files)], dtype=torch.long)
    hour_ids = torch.tensor([
        int(meta_trimmed.iloc[i * N_WINDOWS].get("hour_utc", 0)) % 24
        for i in range(n_files)], dtype=torch.long)

    with torch.no_grad():
        logits = model(emb_f, log_f, site_ids=site_ids, hours=hour_ids)

    probs = sigmoid(logits.numpy().reshape(-1, N_CLASSES))
    pseudo_Y = (probs > threshold).astype(np.uint8)

    n_pseudo_pos = pseudo_Y.sum()
    n_pseudo_files = (pseudo_Y.reshape(n_files, N_WINDOWS, N_CLASSES).sum(axis=1) > 0).sum()
    if verbose:
        print(f"Pseudo-labels: {n_pseudo_pos} positive labels across {n_pseudo_files} files")

    return pseudo_Y[:trim]


def train_with_pseudo_labels(emb_full, scores_full, Y_full, meta_full,
                             n_epochs=40, patience=8, lr=1e-3, verbose=False):
    """[2024 1st] Two-round training with pseudo-labeling.
    Round 1: Train on labeled data only.
    Round 2: Sample a small subset of unlabeled files, generate
    high-confidence pseudo-labels, retrain on combined.
    Designed to be lightweight: max 150 unlabeled files, aggressive gc."""
    n_rounds = CFG["proto_ssm_train"]["pseudo_label_rounds"]
    threshold = CFG["proto_ssm_train"]["pseudo_label_threshold"]
    MAX_PSEUDO_FILES = 150  # cap to avoid OOM

    # Round 1: Train on labeled data
    print(f"\n{'='*60}")
    print(f"ProtoSSM Training — Round 1 (labeled data only)")
    print(f"{'='*60}")
    t0 = time.time()
    model, site2i = train_light_proto_ssm(
        emb_full, scores_full, Y_full, meta_full,
        n_epochs=n_epochs, patience=patience, lr=lr,
        verbose=verbose)
    print(f"Round 1 done in {time.time()-t0:.1f}s")

    # Round 2: Lightweight pseudo-labeling on a small subset
    if n_rounds > 1 and CFG["proto_ssm_train"]["pseudo_label_use_soundscapes"]:
        all_sc_files = sorted((BASE / "train_soundscapes").glob("*.ogg"))
        unlabeled_files = [f for f in all_sc_files
                           if f.stem not in set(full_files)]

        if len(unlabeled_files) > 0:
            # Sample a small random subset to keep memory low
            rng = np.random.RandomState(42)
            rng.shuffle(unlabeled_files)
            subset_files = unlabeled_files[:MAX_PSEUDO_FILES]

            print(f"\n{'='*60}")
            print(f"ProtoSSM Training — Round 2 (pseudo-labeling)")
            print(f"Using {len(subset_files)}/{len(unlabeled_files)} "
                  f"unlabeled files (capped at {MAX_PSEUDO_FILES})")
            print(f"{'='*60}")

            # Run Perch on the small subset
            t0 = time.time()
            unlab_meta, unlab_scores, unlab_embs = run_perch(
                subset_files,
                batch_files=min(CFG["batch_files"], 8),
                verbose=True)

            # Generate pseudo-labels (high confidence only)
            pseudo_Y = generate_pseudo_labels(
                model, unlab_embs, unlab_scores, unlab_meta,
                threshold=threshold, verbose=True)

            # Drop files with zero pseudo-labels (no signal)
            n_files_pseudo = len(subset_files)
            file_has_label = pseudo_Y.reshape(
                n_files_pseudo, N_WINDOWS, N_CLASSES).sum(axis=(1, 2)) > 0
            keep_mask = np.repeat(file_has_label, N_WINDOWS)

            if keep_mask.sum() == 0:
                print("No confident pseudo-labels — skipping Round 2")
                del unlab_embs, unlab_scores, unlab_meta, pseudo_Y
                gc.collect()
                return model, site2i

            unlab_embs_keep = unlab_embs[keep_mask]
            unlab_sc_keep   = unlab_scores[keep_mask]
            pseudo_Y_keep   = pseudo_Y[keep_mask]
            unlab_meta_keep = unlab_meta.iloc[
                [i for i in range(len(keep_mask)) if keep_mask[i]]].reset_index(drop=True)

            n_kept = file_has_label.sum()
            print(f"Retained {n_kept}/{n_files_pseudo} files "
                  f"with pseudo-labels")

            # Combine labeled + pseudo-labeled
            combined_emb  = np.concatenate([emb_full, unlab_embs_keep], axis=0)
            combined_sc   = np.concatenate([scores_full, unlab_sc_keep], axis=0)
            combined_Y    = np.concatenate([Y_full, pseudo_Y_keep], axis=0)
            combined_meta = pd.concat([meta_full, unlab_meta_keep],
                                     ignore_index=True)

            print(f"Combined: {combined_emb.shape[0]} windows "
                  f"(labeled: {emb_full.shape[0]}, "
                  f"pseudo: {unlab_embs_keep.shape[0]})")

            # Retrain on combined data (fewer epochs)
            t0 = time.time()
            model, site2i = train_light_proto_ssm(
                combined_emb, combined_sc, combined_Y, combined_meta,
                n_epochs=max(n_epochs // 2, 15),
                patience=max(patience // 2, 4),
                lr=lr, verbose=verbose)
            print(f"Round 2 done in {time.time()-t0:.1f}s")

            # Aggressive cleanup
            del (unlab_embs, unlab_scores, unlab_meta, pseudo_Y,
                 unlab_embs_keep, unlab_sc_keep, pseudo_Y_keep,
                 unlab_meta_keep, combined_emb, combined_sc,
                 combined_Y, combined_meta)
            gc.collect()
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
        else:
            print("No unlabeled files found — skipping pseudo-labeling")

    return model, site2i


print("✅ ProtoSSM training with pseudo-labeling defined")
print(f"  [2024 1st] pseudo_label_rounds={CFG['proto_ssm_train']['pseudo_label_rounds']}")
print(f"  [2024 1st] pseudo_label_threshold={CFG['proto_ssm_train']['pseudo_label_threshold']}")


In [ ]:
# ── Cell 14: ResidualSSM Model & Training ──────────────────────────
# [2026] Second-pass error correction with zero-init output head.
# Learns to predict (Y - sigmoid(first_pass)) residuals.


class ResidualSSM(nn.Module):
    """Lightweight second-pass model for systematic error correction.
    Output head initialized to zero — corrections start small."""
    def __init__(self, d_input=1536, d_scores=234,
                 d_model=64, d_state=8,
                 n_classes=234, n_windows=12,
                 dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.n_classes = n_classes

        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))

        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        self.ssm_fwd   = SelectiveSSM(d_model, d_state)
        self.ssm_bwd   = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm  = nn.LayerNorm(d_model)
        self.ssm_drop  = nn.Dropout(dropout)

        self.output_head = nn.Linear(d_model, n_classes)
        # Zero init — corrections start at zero
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        x = torch.cat([emb, first_pass], dim=-1)
        h = self.input_proj(x) + self.pos_enc[:, :T, :]

        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                 self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta.unsqueeze(1)

        res = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h   = self.ssm_drop(self.ssm_merge(torch.cat([h_f, h_b], dim=-1)))
        h   = self.ssm_norm(h + res)
        return self.output_head(h)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_residual_ssm(emb_full, first_pass_flat, Y_full,
                       site_ids, hour_ids,
                       n_epochs=30, patience=8, lr=1e-3,
                       correction_weight=0.30, verbose=False):
    """Train ResidualSSM on (Y - sigmoid(first_pass)) residuals."""
    n_files = len(emb_full) // N_WINDOWS
    emb_f = emb_full.reshape(n_files, N_WINDOWS, -1)
    fp_f  = first_pass_flat.reshape(n_files, N_WINDOWS, -1)
    lab_f = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    # Residual target
    fp_prob   = sigmoid(fp_f)
    residuals = lab_f - fp_prob

    print(f"Residuals: mean={residuals.mean():.4f}  std={residuals.std():.4f}")

    n_val = max(1, int(n_files * 0.15))
    rng   = torch.Generator(); rng.manual_seed(42)
    perm  = torch.randperm(n_files, generator=rng).numpy()
    val_i = perm[:n_val]; train_i = perm[n_val:]

    emb_t  = torch.tensor(emb_f,     dtype=torch.float32)
    fp_t   = torch.tensor(fp_f,      dtype=torch.float32)
    res_t  = torch.tensor(residuals, dtype=torch.float32)
    site_t = torch.tensor(site_ids,  dtype=torch.long)
    hour_t = torch.tensor(hour_ids,  dtype=torch.long)

    model = ResidualSSM(n_classes=N_CLASSES)
    print(f"ResidualSSM params: {model.count_parameters():,}")

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")

    best_loss, best_state, wait = float("inf"), None, 0
    for ep in range(n_epochs):
        model.train()
        corr = model(emb_t[train_i], fp_t[train_i],
                     site_ids=site_t[train_i], hours=hour_t[train_i])
        loss = F.mse_loss(corr, res_t[train_i])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        model.eval()
        with torch.no_grad():
            val_corr = model(emb_t[val_i], fp_t[val_i],
                             site_ids=site_t[val_i], hours=hour_t[val_i])
            val_loss = F.mse_loss(val_corr, res_t[val_i])

        if val_loss.item() < best_loss:
            best_loss  = val_loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    model.load_state_dict(best_state)
    print(f"ResidualSSM trained — best val MSE={best_loss:.6f}")
    return model, correction_weight


print("✅ ResidualSSM defined (~439K params, ~20s training)")


In [ ]:
# ── Cell 15: ProtoSSM Inference with Circular-Shift TTA ─────────────
# [2026] Average ProtoSSM predictions across 5 time shifts


def run_tta_proto(proto_model, emb_files, sc_files,
                  site_t, hour_t, shifts=[0, 1, -1, 2, -2]):
    """[2026] TTA by circular-shifting 12-window sequences.
    For each shift s:
      1. Roll embeddings and perch logits by s windows
      2. Run ProtoSSM → get predictions
      3. Roll predictions back by -s (undo shift)
    Average all predictions across shifts."""
    proto_model.eval()
    all_preds = []

    emb_t = torch.tensor(emb_files, dtype=torch.float32)
    sc_t  = torch.tensor(sc_files,  dtype=torch.float32)

    for shift in shifts:
        if shift == 0:
            e_shifted = emb_t
            s_shifted = sc_t
        else:
            e_shifted = torch.roll(emb_t, shift, dims=1)
            s_shifted = torch.roll(sc_t,  shift, dims=1)

        with torch.no_grad():
            out = proto_model(
                e_shifted, s_shifted,
                site_ids=site_t, hours=hour_t
            ).numpy()   # (n_files, 12, 234)

        if shift != 0:
            out = np.roll(out, -shift, axis=1)

        all_preds.append(out)

    return np.mean(all_preds, axis=0)  # (n_files, 12, 234)


print("✅ ProtoSSM TTA with 5 circular shifts defined")


In [ ]:
# ── Cell 16: Complete Inference Pipeline ───────────────────────────
# Step-by-step pipeline combining ALL techniques:
#   A. LightProtoSSM inference with TTA
#   B. Prior tables applied to Perch scores
#   C. MLP probes on prior-adjusted scores
#   D. First-pass ensemble (50% ProtoSSM + 50% MLP-adjusted)
#   E. ResidualSSM correction (weight=0.30)
#   F. Temperature scaling
#   G. Sigmoid
#   H. Post-processing chain: file-confidence → rank-aware →
#      moving_avg → adaptive_delta → gaussian → clip
#   I. SED branch predictions
#   J. Final rank-average ensemble with power scaling


def build_site_hour_ids(meta_df, site2i, n_sites=20):
    """Build site and hour ID arrays from metadata.
    Uses positional grouping (every N_WINDOWS rows = 1 file)"""
    n_files = len(meta_df) // N_WINDOWS
    trim = n_files * N_WINDOWS
    meta_trimmed = meta_df.iloc[:trim]
    site_ids = np.zeros(n_files, dtype=np.int64)
    hour_ids = np.zeros(n_files, dtype=np.int64)
    for i in range(n_files):
        row = meta_trimmed.iloc[i * N_WINDOWS]
        site_ids[i] = min(site2i.get(str(row.get("site", "")), 0), n_sites - 1)
        hour_ids[i] = int(row.get("hour_utc", 0)) % 24
    return site_ids, hour_ids


def run_full_pipeline(emb_tr, sc_tr, Y_tr, meta_tr,
                      emb_te, sc_te, meta_te, test_paths):
    """Complete inference pipeline with all integrated techniques."""
    n_tr_files = len(emb_tr) // N_WINDOWS
    n_te_files = len(emb_te) // N_WINDOWS

    # ── Step A: Train LightProtoSSM (with pseudo-labeling) ─────────
    print("\n" + "="*60)
    print("STEP A: Train LightProtoSSM")
    print("="*60)
    t0 = time.time()
    proto_model, site2i_tr = train_with_pseudo_labels(
        emb_tr, sc_tr, Y_tr, meta_tr,
        n_epochs=CFG["proto_ssm_train"]["n_epochs"],
        patience=CFG["proto_ssm_train"]["patience"],
        lr=CFG["proto_ssm_train"]["lr"],
        verbose=CFG["verbose"])
    print(f"ProtoSSM training: {time.time()-t0:.1f}s")

    # ── Step A2: ProtoSSM TTA inference on TEST ─────────────────────
    print("\n" + "="*60)
    print("STEP A2: ProtoSSM TTA Inference")
    print("="*60)
    emb_te_f = emb_te.reshape(n_te_files, N_WINDOWS, -1)
    sc_te_f  = sc_te.reshape(n_te_files, N_WINDOWS, -1)

    te_site_ids, te_hour_ids = build_site_hour_ids(meta_te, site2i_tr)
    tr_site_ids, tr_hour_ids = build_site_hour_ids(meta_tr, site2i_tr)

    proto_te_out = run_tta_proto(
        proto_model, emb_te_f, sc_te_f,
        site_t=torch.tensor(te_site_ids, dtype=torch.long),
        hour_t=torch.tensor(te_hour_ids, dtype=torch.long),
        shifts=[0, 1, -1, 2, -2])
    proto_scores_te = proto_te_out.reshape(-1, N_CLASSES).astype(np.float32)

    # Also get training predictions for residual model
    emb_tr_f = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
    sc_tr_f  = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)
    proto_tr_out = run_tta_proto(
        proto_model, emb_tr_f, sc_tr_f,
        site_t=torch.tensor(tr_site_ids, dtype=torch.long),
        hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
        shifts=[0, 1, -1, 2, -2])
    proto_scores_tr = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)
    print(f"ProtoSSM TTA inference done")

    # ── Step B: Prior tables ────────────────────────────────────────
    print("\n" + "="*60)
    print("STEP B: Build & Apply Prior Tables")
    print("="*60)
    prior_tables = build_prior_tables(sc, Y_SC)
    sc_te_prior = apply_prior(
        sc_te, sites=meta_te["site"].to_numpy(),
        hours=meta_te["hour_utc"].to_numpy(),
        tables=prior_tables, lambda_prior=CFG["prior_lambda"])
    sc_tr_prior = apply_prior(
        sc_tr, sites=meta_tr["site"].to_numpy(),
        hours=meta_tr["hour_utc"].to_numpy(),
        tables=prior_tables, lambda_prior=CFG["prior_lambda"])
    print(f"Priors applied (lambda={CFG['prior_lambda']})")

    # ── Step C: MLP probes ──────────────────────────────────────────
    print("\n" + "="*60)
    print("STEP C: Train & Apply MLP Probes")
    print("="*60)
    t0 = time.time()
    probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
        emb=emb_tr, scores_raw=sc_tr_prior, Y=Y_tr,
        min_pos=5, pca_dim=64, alpha_blend=CFG["mlp_alpha_blend"])

    sc_te_mlp = apply_mlp_probes_vectorized(
        emb_te, sc_te_prior, probe_models, emb_scaler, emb_pca, alpha_blend)
    sc_tr_mlp = apply_mlp_probes_vectorized(
        emb_tr, sc_tr_prior, probe_models, emb_scaler, emb_pca, alpha_blend)
    print(f"MLP probes done in {time.time()-t0:.1f}s")

    # ── Step D: First-pass ensemble ────────────────────────────────
    print("\n" + "="*60)
    print("STEP D: First-Pass Ensemble (ProtoSSM + MLP)")
    print("="*60)
    ew = CFG["first_pass_ensemble_w"]
    first_pass_te = ew * proto_scores_te + (1.0 - ew) * sc_te_mlp
    first_pass_tr = ew * proto_scores_tr + (1.0 - ew) * sc_tr_mlp
    print(f"Ensemble: {ew:.0%} ProtoSSM + {1-ew:.0%} MLP")

    # ── Step E: ResidualSSM correction ──────────────────────────────
    print("\n" + "="*60)
    print("STEP E: Train & Apply ResidualSSM")
    print("="*60)
    t0 = time.time()
    res_model, corr_w = train_residual_ssm(
        emb_full=emb_tr, first_pass_flat=first_pass_tr, Y_full=Y_tr,
        site_ids=tr_site_ids, hour_ids=tr_hour_ids,
        n_epochs=CFG["residual_ssm"]["n_epochs"],
        patience=CFG["residual_ssm"]["patience"],
        correction_weight=CFG["residual_ssm"]["correction_weight"])

    # Apply correction to TEST
    fp_te_f = first_pass_te.reshape(n_te_files, N_WINDOWS, -1)
    res_model.eval()
    with torch.no_grad():
        correction = res_model(
            torch.tensor(emb_te_f, dtype=torch.float32),
            torch.tensor(fp_te_f,  dtype=torch.float32),
            site_ids=torch.tensor(te_site_ids, dtype=torch.long),
            hours   =torch.tensor(te_hour_ids, dtype=torch.long),
        ).numpy()
    correction_flat = correction.reshape(-1, N_CLASSES).astype(np.float32)
    final_logits_te = first_pass_te + corr_w * correction_flat
    print(f"ResidualSSM done in {time.time()-t0:.1f}s (weight={corr_w:.2f})")

    # ── Step F: Temperature scaling ─────────────────────────────────
    print("\n" + "="*60)
    print("STEP F: Per-Taxon Temperature Scaling")
    print("="*60)
    final_logits_te = per_taxon_temperature(final_logits_te)
    print(f"Birds T={CFG['bird_temp']}, Texture T={CFG['texture_temp']}")

    # ── Step G: Sigmoid → probabilities ─────────────────────────────
    probs = sigmoid(final_logits_te)
    del final_logits_te, correction, correction_flat, fp_te_f
    gc.collect()

    # ── Step H: Post-processing chain ───────────────────────────────
    print("\n" + "="*60)
    print("STEP H: Post-Processing Chain")
    print("="*60)

    # H1: File-level confidence scaling [2026]
    probs = file_confidence_scale(
        probs, n_windows=N_WINDOWS,
        top_k=CFG["file_conf_top_k"], power=CFG["file_conf_power"])
    print("  H1: File-confidence scaling applied")

    # H2: Rank-aware scaling [2026]
    probs = rank_aware_scaling(
        probs, n_windows=N_WINDOWS, power=CFG["rank_aware_power"])
    print("  H2: Rank-aware scaling applied")

    # H3: Moving average smoothing [2024 1st]
    if CFG["moving_avg_smooth"]:
        probs = moving_average_smooth(
            probs, n_windows=N_WINDOWS, kernel_size=CFG["moving_avg_kernel"])
        print(f"  H3: Moving-average smooth (kernel={CFG['moving_avg_kernel']})")

    # H4: Adaptive delta smoothing [2026]
    probs = adaptive_delta_smooth(
        probs, n_windows=N_WINDOWS,
        base_alpha=CFG["adaptive_delta_base_alpha"])
    print("  H4: Adaptive delta smoothing applied")

    # H5: Gaussian smoothing [2025 1st]
    if CFG["gaussian_smooth"]:
        probs = gaussian_smooth(probs, axis=0)
        print("  H5: Gaussian smoothing applied")

    # H6: Clip to [0, 1]
    probs = np.clip(probs, 0.0, 1.0)
    print("  H6: Clipped to [0, 1]")

    # Save ProtoSSM-only submission for ensemble
    protossm_sub = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
    protossm_sub.insert(0, "row_id", meta_te["row_id"].values)
    protossm_sub.to_csv("submission_protossm.csv", index=False)
    print(f"  ProtoSSM submission saved: {protossm_sub.shape}")

    # ── Step I: SED branch predictions ──────────────────────────────
    print("\n" + "="*60)
    print("STEP I: SED Branch Inference")
    print("="*60)
    t0 = time.time()
    sed_rows, sed_preds = run_sed_inference(test_paths, verbose=True)
    print(f"SED inference: {time.time()-t0:.1f}s")

    sed_sub = pd.DataFrame(np.clip(sed_preds, 0.0, 1.0), columns=PRIMARY_LABELS)
    sed_sub.insert(0, "row_id", sed_rows)
    sed_sub.to_csv("submission_sed.csv", index=False)
    print(f"  SED submission saved: {sed_sub.shape}")

    # ── Step J: Final rank-average ensemble with power scaling ──────
    print("\n" + "="*60)
    print("STEP J: Final Ensemble (Rank-Average + Power Scaling)")
    print("="*60)

    # Align rows
    sed_sub_aligned = sed_sub.set_index("row_id").loc[protossm_sub["row_id"]].reset_index()
    cols = [c for c in protossm_sub.columns if c != "row_id"]

    EPS = 1e-5
    pa = np.clip(protossm_sub[cols].to_numpy(np.float32), EPS, 1 - EPS)
    pb = np.clip(sed_sub_aligned[cols].to_numpy(np.float32), EPS, 1 - EPS)

    if CFG["ensemble_power"] != 1.0:
        # [2025 1st] Power scaling before rank-average
        blended = power_scale_blend(
            pa, pb,
            CFG["proto_weight"], CFG["sed_weight"],
            power=CFG["ensemble_power"])
        print(f"  Power-scaled blend (power={CFG['ensemble_power']})")
    else:
        # [2026] Standard rank-average
        ra = pd.DataFrame(pa).rank(axis=0, pct=True).to_numpy(np.float32)
        rb = pd.DataFrame(pb).rank(axis=0, pct=True).to_numpy(np.float32)
        blended = CFG["proto_weight"] * ra + CFG["sed_weight"] * rb
        print(f"  Rank-average blend")

    final_sub = protossm_sub.copy()
    final_sub[cols] = np.clip(blended, 0.0, 1.0).astype(np.float32)

    print(f"\n{'='*60}")
    print(f"FINAL SUBMISSION: {final_sub.shape}")
    print(f"Blend: ProtoSSM {CFG['proto_weight']:.0%} + SED {CFG['sed_weight']:.0%}")
    print(f"Total wall time: {(time.time() - _WALL_START)/60:.1f} min")
    print(f"{'='*60}")

    # Cleanup
    del proto_model, res_model, probe_models
    gc.collect()

    return final_sub


print("✅ Full inference pipeline defined")
print("  Steps: A(ProtoSSM+TTA) → B(Priors) → C(MLP) → D(Ensemble) →")
print("         E(ResidualSSM) → F(TempScale) → G(Sigmoid) →")
print("         H(PostProc) → I(SED) → J(FinalBlend)")


In [ ]:
# ── Cell 17: Submission ─────────────────────────────────────────────

# Load test files
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))

if not test_paths:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test — dry-run on {n} train files")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print(f"Hidden test files: {len(test_paths)}")

# Run Perch inference on test files
print("\nRunning Perch inference on test files…")
t0 = time.time()
meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=True)
print(f"Perch test inference: {time.time()-t0:.1f}s  shape={sc_te.shape}")

# Run full pipeline
final_sub = run_full_pipeline(
    emb_tr=emb_tr, sc_tr=sc_tr, Y_tr=Y_FULL_aligned, meta_tr=meta_tr,
    emb_te=emb_te, sc_te=sc_te, meta_te=meta_te, test_paths=test_paths)

# Save final submission
final_sub.to_csv("submission.csv", index=False)
print(f"\n✅ submission.csv saved — shape {final_sub.shape}")
print(f"Total wall time: {(time.time() - _WALL_START)/60:.1f} min")

# Verify submission format
assert list(final_sub.columns) == ["row_id"] + PRIMARY_LABELS
assert len(final_sub) == len(test_paths) * N_WINDOWS
assert not final_sub.isna().any().any()
print("\n✅ Submission verified — all checks passed")

# Memory cleanup
gc.collect()
print("Memory freed. Done!")
